In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import time
import pickle
from lightning_modelling.deter_architecture import Unet, FullyConnectedNet_1d
from lightning_modelling.models import XGBoostModel, LogisticRegressionModel, GAMModel
from lightning_modelling.dataset import CustomPTDataset, create_train_test, get_seasons
from lightning_modelling.metrics import Metrics
from lightning_modelling.trainer import Trainer
from lightning_modelling.common_path import DATASET_PATH, MODELS_PATH

## Load data

In [ ]:
ALL_YEARS = list(range(2008, 2024))
HELD_OUT_YEARS = [2008, 2015, 2023]
ALL_LOYO_YEARS = [year for year in ALL_YEARS if year not in HELD_OUT_YEARS]
TRAIN_YEARS = ALL_LOYO_YEARS
TEST_YEARS = HELD_OUT_YEARS
SCALER_PATH = os.path.join(DATASET_PATH, "scaler", "scaler_full.pkl")

print("Creating the dataset and dataloader..............")
_, _, TEST_DATASET = create_train_test(DATASET_PATH, TRAIN_YEARS, TEST_YEARS, scaler_path=SCALER_PATH)
TEST_DATASET.metadata_csv["year"] = pd.to_datetime(TEST_DATASET.metadata_csv["date"]).dt.year

extreme_days = pd.read_csv(DATASET_PATH / "extreme_days_top_0.05.csv")
all_extremes_metadata = TEST_DATASET.metadata_csv[TEST_DATASET.metadata_csv["date"].isin(extreme_days["date"])]
test_extremes = all_extremes_metadata[all_extremes_metadata["year"].isin(TEST_YEARS)]
test_extremes_ids = test_extremes["id"].values
TEST_EXTREMES_DATASET = CustomPTDataset(root_dir=DATASET_PATH, sample_ids=test_extremes_ids, scaler_path=SCALER_PATH)

TEST_EXTREMES_LOADER = DataLoader(TEST_EXTREMES_DATASET, batch_size=1, shuffle=False)

ALL_DAYS_SEASONS = np.array(get_seasons(ALL_YEARS))
EXTREMES_SEASONS = ALL_DAYS_SEASONS[test_extremes_ids]

## Load models

In [ ]:
CHANNELS = [16, 32, 64]
NUM_RESIDUAL_LAYERS = 2
RECALIBRATION = 'platt_scaling'

# Initialize unet
unet = Unet(
    channels = CHANNELS,
    num_residual_layers = NUM_RESIDUAL_LAYERS,
    name = "unet",
    recalibration_method = RECALIBRATION,
)

unet.load_state_dict(torch.load(MODELS_PATH / 'unet.pth', map_location=torch.device('cpu')))
unet.eval()

xgb_name = "xgb"
xgb_model = pickle.load(open(MODELS_PATH / "xgb.pkl", "rb"))
xgb = XGBoostModel(model=xgb_model, name=xgb_name, remove_vars=None)


gam_name = "gam"
gam_model = pickle.load(open(MODELS_PATH / "gam.pkl", "rb"))
gam = GAMModel(model=gam_model, name=gam_name, remove_vars=None)


mlp_name = "mlp"
HIDDEN_DIMS = [16, 32, 16]

mlp = FullyConnectedNet_1d(
    name = mlp_name,
    save_path = None,
    hidden_dims = HIDDEN_DIMS,
    recalibration_method = RECALIBRATION,
    removed_features = [],
)

mlp.load_state_dict(torch.load(MODELS_PATH / "mlp.pth", map_location=torch.device('cpu')))
mlp.eval()

logreg_name = "logreg"
logreg_model = pickle.load(open(MODELS_PATH / "log_reg.pkl", "rb"))
logreg = LogisticRegressionModel(model=logreg_model, name="logreg", remove_vars=None)

models = [logreg, gam, xgb, mlp, unet]


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EARLY_STOP = True
N_EARLY = 2
LOYO_ID = 100

trainers = []
for model in models:
    trainers.append(
        Trainer(model = model,
            full_dataset = None,
            train_dataloader = None,
            calibration_dataloader = None,
            test_dataloader = None,
            test_extremes_dataloader = TEST_EXTREMES_LOADER,
            seasons = EXTREMES_SEASONS,
            device = DEVICE,
            data_path = None,
            eval_path = None,
            n_years = len(TEST_YEARS),
            calibration_early_stopping = True,
            test_early_stopping = EARLY_STOP,
            n_early = N_EARLY,
            )
    )

In [ ]:
for j, trainer in enumerate(trainers):
    print(f"Testing model {trainer.model.name} on the extreme event days..............")
    trainer.model.to(trainer.device)
    trainer.model.eval()
    loader = trainer.test_extremes_dataloader
    if trainer.test_early_stopping:
        n_batches = min(trainer.n_early, len(loader))
    else:
        n_batches = len(loader)
    pred_events_maps = []
    trainer.metrics = Metrics(kernel_size=1, save_path=trainer.eval_path, reduction='mean', n_batches=n_batches, extremes=True)
    print(f"Testing on {n_batches} events........")
    T, C, H, W  = TEST_DATASET[0].shape
    with torch.no_grad():
        # pbar = tqdm(enumerate(loader), total=n_batches)
        start_time_model = time.time()
        for i, sample in enumerate(TEST_DATASET):
            if trainer.test_early_stopping and i >= trainer.n_early: 
                break
            if trainer.seasons is not None:
                season = trainer.seasons[i] 
            if i+1%100 == 0 :
                print(f"{i}/{min(n_batches, len(loader))}, test duration : {time.time() - start_time_model}")
            batch = sample.view(T, C, H, W)     # shape (T, c+2, h, w)
            x = batch[:, :-1, :, :].to(trainer.device)
            y_true = batch[:, -1, :, :].to(trainer.device)
            y_true = (y_true >= 2).float()
            pred = trainer.model(x)
            trainer.metrics.update_all(y_true, pred, season)

        trainer.metrics.compute_all()
    print()
    print("="*50)
    print()

## Print metrics

In [ ]:
print("METRICS COMPUTED FOR ALL MODELS ON EXTREME DAYS")
print()
print("="*40)       
print()
for trainer in trainers:
    print(f"Model: {trainer.model.name}")
    print(f"ROC AUC: {trainer.metrics.auc_calc.roc_auc:.3f}                  | PR AUC: {trainer.metrics.auc_calc.ap:.3f}")
    print(f"Deviance: {trainer.metrics.deviance_score.score:.3f}                 | Dice: {trainer.metrics.iou.iou:.3f}")
    print(f"FBSS: {trainer.metrics.fractional_scores.fbss:.3f}")
    print()
    print("="*40)       
    print()     